In [1]:
# 1B: CSV -> Parquet with small dtypes (one-time)
import pandas as pd, numpy as np, time
from pathlib import Path

ROOT = Path.home() / "Documents/uplift"
RAW = ROOT / "data/raw/criteo-uplift-v2.1.csv.gz"
PQ = ROOT / "data/processed/criteo.parquet"
F = [f"f{i}" for i in range(12)]

dtypes = {c: "float32" for c in F} | {c: "int8" for c in ["treatment", "conversion", "visit", "exposure"]}
t0 = time.time()
df = pd.read_csv(RAW, dtype=dtypes)
df.to_parquet(PQ, index=False)
print(df.shape, f"| {df.memory_usage().sum() / 1e9:.2f} GB in memory | {time.time() - t0:.0f}s")

(13979592, 16) | 0.73 GB in memory | 12s


In [2]:
## STEP 1C AUDIT 
df = pd.read_parquet(PQ)
print("missing values:", int(df.isna().sum().sum()))
print(df[["treatment", "exposure", "visit", "conversion"]].mean().round(4))
print(pd.crosstab(df["treatment"], df["exposure"], margins = True))

#outcome rate by arm
print(df.groupby("treatment")[["visit","conversion"]].agg(["mean", "size"]).round(5))

# randomization check standarized mean differnce smd per feature
m = df.groupby("treatment")[F].mean()
v = df.groupby("treatment")[F].var()

smd = (m.loc[1] - m.loc[0]) / np.sqrt((v.loc[1] + v.loc[0]) /2)
print("SMD (treated vs control):\n", smd.round(4).to_string())
print(df[F].nunique().to_string())

missing values: 0
treatment     0.8500
exposure      0.0306
visit         0.0470
conversion    0.0029
dtype: float64
exposure          0       1       All
treatment                            
0           2096937       0   2096937
1          11454443  428212  11882655
All        13551380  428212  13979592
             visit           conversion          
              mean      size       mean      size
treatment                                        
0          0.03820   2096937    0.00194   2096937
1          0.04854  11882655    0.00309  11882655
SMD (treated vs control):
 f0    -0.0069
f1     0.0232
f2    -0.0064
f3    -0.0489
f4     0.0068
f5    -0.0295
f6    -0.0405
f7     0.0208
f8    -0.0229
f9     0.0242
f10    0.0092
f11   -0.0052
f0     2181959
f1          60
f2      808663
f3         552
f4         260
f5         132
f6        1645
f7      622143
f8        3743
f9        1594
f10     483634
f11        136


In [14]:
# visit rate by f3 decile, control group only (no ad effect)
c = df[df["treatment"] == 0]
print(c.groupby(pd.qcut(c["f3"], 10, duplicates="drop"), observed=True)["visit"].mean().round(4))
print(c["f3"].value_counts(normalize=True).head(5).round(3))

f3
(-8.399, 2.935]    0.1025
(2.935, 4.68]      0.0305
Name: visit, dtype: float64
f3
4.679882    0.830
3.907662    0.041
3.359763    0.022
2.934780    0.015
2.587543    0.011
Name: proportion, dtype: float64


In [3]:
import pandas as pd, numpy as np
from pathlib import Path
from scipy import stats
import statsmodels.api as sm

t = df["treatment"].to_numpy()

In [7]:
#STEP 2A
#intent to treat effect difference in proportions

def ab_diff(y,t):
    y1,y0 = y[t==1], y[t==0]
    p1,p0 = y1.mean(), y0.mean()
    se = np.sqrt(p1*(1-p1) / len(y1) + p0 * (1-p0) / len(y0))
    d = p1-p0
    return dict(p0 = p0, p1 = p1, diff = d, se = se , lo = d - 1.96 * se, hi = d + 1.96 * se, rel = d / p0, z = d/ se)
res = {k: ab_diff(df[k].to_numpy(),t) for k in ["visit", "conversion"]}

for k, r in res.items():
    print(f"{k:10s} control {r['p0']:.4%} | treated {r['p1']:.4%} | lift {r['diff']*100:+.3f} pts "
          f"[{r['lo']*100:.3f}, {r['hi']*100:.3f}] | relative {r['rel']:+.1%} | z = {r['z']:.0f}")
    


visit      control 3.8201% | treated 4.8543% | lift +1.034 pts [1.006, 1.063] | relative +27.1% | z = 71
conversion control 0.1938% | treated 0.3089% | lift +0.115 pts [0.108, 0.122] | relative +59.4% | z = 34


In [10]:
#STEP 2B REGRESSION ADJUSTMENT  DOES THE LIFT SURVIVE CONTROLLING FOR THE FEATURE
Xr = sm.add_constant(df[["treatment"] + F].astype("float64"))

for k in ["visit", "conversion"]:
    fit = sm.OLS(df[k].astype("float64"), Xr).fit(cov_type="HC1")
    b, se = fit.params["treatment"], fit.bse["treatment"]
    print(f"{k:10s} adjusted {b*100:+.3f} pts [{(b-1.96*se)*100:.3f}, {(b+1.96*se)*100:.3f}] "
        f"| raw {res[k]['diff']*100:+.3f} | SE raw {res[k]['se']*100:.4f} vs adjusted {se*100:.4f}")
del Xr

visit      adjusted +0.698 pts [0.673, 0.723] | raw +1.034 | SE raw 0.0146 vs adjusted 0.0126
conversion adjusted +0.092 pts [0.085, 0.098] | raw +0.115 | SE raw 0.0034 vs adjusted 0.0032


In [12]:
#STEP 2B-2 randomization check with a model (AUC = .5 IS RANDOM)
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

s = df.sample(2_000_000, random_state = 0).reset_index(drop = True)
clf = lgb.LGBMClassifier(n_estimators = 200,  learning_rate = .1, num_leaves = 63, verbose = -1)
e = cross_val_predict(clf, s[F], s["treatment"], cv = StratifiedKFold(5, shuffle = True, random_state = 0),
                      method = "predict_proba")[:,1]
print(f"propensity AUC: {roc_auc_score(s['treatment'], e):.3f}")
print(pd.Series(e).describe(percentiles=[.01, .05, .5, .95, .99]).round(3))

propensity AUC: 0.508
count    2000000.000
mean           0.850
std            0.013
min            0.168
1%             0.826
5%             0.843
50%            0.849
95%            0.869
99%            0.908
max            0.996
dtype: float64


In [18]:
# STEP 2B-3 AIPW: outcome models + propsensity, robust ifeither one is right 

y = s["visit"].to_numpy(); T = s["treatment"].to_numpy()
m1 = np.zeros(len(s)); m0 = np.zeros(len(s))
for tr, te in StratifiedKFold(5, shuffle = True, random_state =1).split(s,T):
    Xtr, Ttr, ytr = s[F].iloc[tr], T[tr], y[tr]
    reg = lgb.LGBMClassifier(n_estimators = 200,  learning_rate = 0.1, num_leaves = 63, berbose = -1)
    m1[te] = reg.fit(Xtr[Ttr == 1], ytr[Ttr == 1]).predict_proba(s[F].iloc[te])[:,1]
    m0[te] = reg.fit(Xtr[Ttr == 0], ytr[Ttr == 0]).predict_proba(s[F].iloc[te])[:,1]
ec = np.clip(e, .01, 0.99)
psi = m1 - m0 + T * (y-m1) / ec - (1-T) * (y-m0) / (1 - ec)
raw = y[T == 1].mean() - y[T == 0].mean()
print(f"AIPW visit lift: {psi.mean()*100:+.3f} pts ± {1.96 * psi.std() / np.sqrt(len(psi)) * 100:.3f} "
      f"| raw on this sample {raw*100:+.3f}")


AIPW visit lift: +0.768 pts ± 0.070 | raw on this sample +1.038


In [21]:
## STEP 2C EFFECT ON THE EXPOSED 
share = df.loc[df["treatment"] == 1, "exposure"].mean()
for k, r in res.items():
    cace, se = r["diff"] / share, r["se"] / share
    print(f"{k:10s} effect on exposed: {cace*100:+.2f} pts [{(cace-1.96*se)*100:.2f}, {(cace+1.96*se)*100:.2f}]")

print(f"share of treated users exposed: {share:.4f}")  

aipw, aipw_se = psi.mean(), psi.std() / np.sqrt(len(psi))
print(f"visit effect on exposed (AIPW): {aipw/share*100:+.2f} pts "
      f"[{(aipw - 1.96*aipw_se)/share*100:.2f}, {(aipw + 1.96*aipw_se)/share*100:.2f}]")

visit      effect on exposed: +28.70 pts [27.90, 29.50]
conversion effect on exposed: +3.20 pts [3.01, 3.38]
share of treated users exposed: 0.0360
visit effect on exposed (AIPW): +21.32 pts [19.37, 23.28]


In [22]:
#STEP 2D MINIM DECTEABLE EFFECT MDE AT 80% POWER, ALSPHA .05
def mde(p0, n1, n0, alpha = 0.05, power = .8):
    return (stats.norm.ppf(1-alpha / 2) + stats.norm.ppf(power)) * np.sqrt(p0 * (1-p0) * (1 / n1 + 1  / n0))
n1, n0 = (t==1).sum(), (t==0).sum()
for k, r in res.items():
    for frac in [1.0, 0.01]:
        m = mde(r["p0"], n1 * frac, n0 * frac)
        print(f"{k:10s} {frac:>5.0%} of users: MDE {m*100:.3f} pts ({m / r['p0']:.1%} relative) "
              f"| observed {r['diff']*100:.3f}")


visit       100% of users: MDE 0.040 pts (1.1% relative) | observed 1.034
visit         1% of users: MDE 0.402 pts (10.5% relative) | observed 1.034
conversion  100% of users: MDE 0.009 pts (4.8% relative) | observed 0.115
conversion    1% of users: MDE 0.092 pts (47.6% relative) | observed 0.115


STEP 3A Train/Test SPlit

In [25]:
#STEP 3A train/test split
from sklearn.model_selection import train_test_split
strat = df["treatment"].astype(str) + df["visit"].astype(str)
tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size = 0.3, random_state = 42)
Xtr, Xte = df[F].iloc[tr_idx], df[F].iloc[te_idx]
ttr, tte = t[tr_idx], t[te_idx]
ytr, yte = df["visit"].to_numpy()[tr_idx], df["visit"].to_numpy()[te_idx]
print(f"train {len(tr_idx):,} | test {len(te_idx):,} | visit rate {ytr.mean():.4f} / {yte.mean():.4f}")

train 9,785,714 | test 4,193,878 | visit rate 0.0470 / 0.0470


STEP 3B S-LEARNER (ONE MODEL)

In [27]:
#3B - learner: treatment is jsut another feature; uplift = p(visit | x, t=1) - p(visit | x , t=0 )
params = dict(n_estimators = 300, learning_rate =0.05, num_leaves = 63, min_child_samples = 200, verbose = -1)
s_model = lgb.LGBMClassifier(**params).fit(Xtr.assign(treatment = ttr), ytr)
up_s = (s_model.predict_proba(Xte.assign(treatment=1))[:,1] - 
        s_model.predict_proba(Xte.assign(treatment = 0))[:,1])
print(f"S-learner: mean uplift {up_s.mean()*100:+.3f} pts | users with uplift exactly 0: {(up_s == 0).mean():.1%}")

S-learner: mean uplift +0.716 pts | users with uplift exactly 0: 0.0%


STEP 3C: T-learner(TWO MODELS)

In [29]:
#3C T-LEARNER: SEPARATE MODELS FOR TREATED AND CONTROL ; UPLIFT = DIFFERNCE OF THEIR PREDTIONS

t1 = lgb.LGBMClassifier(**params).fit(Xtr[ttr == 1], ytr[ttr == 1])
t0 = lgb.LGBMClassifier(**params).fit(Xtr[ttr == 0], ytr[ttr == 0])

p1, p0 = t1.predict_proba(Xte)[:,1], t0.predict_proba(Xte)[:,1]
up_t = p1 - p0

print(f"T-learner: mean uplift {up_t.mean()*100:+.3f} pts | 1%-99%: "
      f"[{np.percentile(up_t, 1)*100:.2f}, {np.percentile(up_t, 99)*100:.2f}]")

T-learner: mean uplift +0.738 pts | 1%-99%: [-2.11, 13.93]


STEP 3D COMAPRE AGAIST RANDOM TARGETING AND AGAINT A COMMON TRAP

In [30]:
# 3D: Qini AUC and uplift in the top k% (higher = better targeting)
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
from sklift.metrics import qini_auc_score, uplift_at_k

def report(name, score):
    q = qini_auc_score(yte, score, tte)
    u10 = uplift_at_k(yte, score, tte, strategy="overall", k =0.1)
    u30 = uplift_at_k(yte, score, tte, strategy="overall", k =0.30)
    print(f"{name:20s} Qini AUC {q:.4f} | uplift top 10%: {u10*100:+.2f} pts | top 30%: {u30*100:+.2f} pts")
    
    
report("random", np.random.default_rng(0).random(len(yte)))
report("S-learner", up_s)
report("T-learner", up_t)
report("p(visit) [the trap]", p1)

random               Qini AUC 0.0009 | uplift top 10%: +1.07 pts | top 30%: +1.06 pts
S-learner            Qini AUC 0.0947 | uplift top 10%: +6.72 pts | top 30%: +2.99 pts
T-learner            Qini AUC 0.0857 | uplift top 10%: +6.18 pts | top 30%: +2.95 pts
p(visit) [the trap]  Qini AUC 0.0861 | uplift top 10%: +5.37 pts | top 30%: +2.94 pts
